In [ ]:
from typing import Any, List, Optional, Callable
from jaxtyping import Array, Float, Int, PyTree # https://github.com/google/jaxtyping

import dataclasses

import jax
import jax.numpy as jnp
import numpy as np
import distrax
print("Distrax version:", distrax.__version__)
import flax
from flax import nnx
print("Flax version:", flax.__version__)

Distrax version: 0.1.8
Flax version: 0.12.7


In [ ]:
from distrax.bijectors import MaskedCoupling, Inverse, Transformed, Chain


class MLP(nnx.Module):
    def __init__(self,
                 layer_dims: Sequence[tuple],
                 activation: Callable = nnx.leaky_relu,
                 use_bias: bool = True,
                 rngs: nnx.Rngs = nnx.Rngs(0),
                 kernel_init: Callable = nnx.initializers.lecun_normal()
                 ):
                 
        assert len(layer_dims)>0, "At least one layer dimension must be specified in layer_dims."
        self.activation = activation
        self.layer_dims = tuple(tuple(x) if not isinstance(x, tuple) else x for x in layer_dims)
        self.layers = nnx.List()
        for layer_dim in layer_dims:
            self.layers.append(nnx.Linear(layer_dim[0], 
                                          layer_dim[1], 
                                          rngs=rngs, 
                                          use_bias=use_bias, 
                                          kernel_init=kernel_init
                                          )
                                )
    def __call__(self, x):
        for l, layer in enumerate(self.layers[:-1]):
            x = self.activation(layer(x))
        x = self.layers[-1](x)
        return x
        

class Conditioner(nnx.Module):
    def __init__(self,
                 features_shape: tuple,
                 context_shape: tuple,
                 hidden_dims: Sequence[tuple],
                 num_bijector_params: int,
                 activation: Callable = nnx.leaky_relu,
                 rngs: nnx.Rngs = nnx.Rngs(0),
                 kernel_init: Callable = nnx.initializers.lecun_normal(),
                 ):
        self.features_shape = features_shape
        self.context_shape = context_shape
        self.num_bijector_params = num_bijector_params
        self.activation = activation

        self.n_flat_features = jax.tree.reduce(lambda carry, x: carry*x, features_shape)
        self.n_flat_context = jax.tree.reduce(lambda carry, x: carry*x, context_shape)``
        self.layer_dims = ((self.n_flat_features+self.n_flat_context, hidden_dims[0][0]),) + hidden_dims

        self._conditioner = nnx.List()
        self._conditioner_mlp = MLP(self.layer_dims, activation, rngs=rngs, kernel_init=kernel_init) # Build the NN as specified by layer_dims that learns the sline transformation parameters
        self._conditioner_out = nnx.Linear(self.layer_dims[-1][-1],
                                            self.n_flat_features*num_bijector_params,
                                            rngs=rngs,
                                            kernel_init=kernel_init
                                            )
    
    def __call__(self, x, context=None):
        # Flatten feature vector to prepare for parsing into spline transform Conditioner, which is a multilayer perceptron
        x_batch_shape = x.shape[:-len(self.features_shape)]
        x.reshape(*x_batch_shape, -1)

        if context is not None:
            # Stack the flattened context vector to the flattened feature vector for the Conditioner in a conditional flow transform
            context_batch_shape = context.shape[:-len(self.context_shape)]
            assert x_batch_shape == context_batch_shape, f"Batch shape mismatch: features (x) has shape {batch_shape}, context has shape {batch_shape_context}"
            context.reshape(*context_batch_shape, -1)
            x = jnp.hstack([context,x])

        x = self._conditioner_mlp(x)
        x = self.activation(x)
        x = self._conditioner_out(x)
        x = x.reshape(*x_batch_shape, *(self.features_shape + (self.num_bijector_params,)))
        return x


class ConditionalInverse(Inverse):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, x: Array, context: Optional[Array] = None) -> Array:
        return self._bijector.inverse(x, context)

    def inverse(self, y: Array, context: Optional[Array] = None) -> Array:
        return self._bijector.forward(y, context)

    def forward_and_log_det(self, x: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        return self._bijector.inverse_and_log_det(x, context)

    def inverse_and_log_det(self, y: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        return self._bijector.forward_and_log_det(y, context)


class ConditionalMaskedCoupling(MaskedCoupling):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, x: Array, context: Optional[Array] = None) -> Array:
        y, _ = self.forward_and_log_det(x)
        return y

    def inverse(self, y: Array, context: Optional[Array] = None) -> Array:
        x, _ = self.inverse_and_log_det(y)
        return x

    def forward_and_log_det(self, x: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        self._check_forward_input_shape(x)
        masked_x = jnp.where(self._event_mask, x, 0.0)
        params = self._conditioner(masked_x, context)
        y0, log_d = self._inner_bijector(params).forward_and_log_det(x)
        y = jnp.where(self._event_mask, x ,y0)
        logdet = math.sum_last(
            jnp.where(self._mask, 0., log_d),
            self._event_ndims - self._inner_event_ndims
            )
        return y, logdet

    def inverse_and_log_det(self, y: Array, context: Optional[Array] = None) -> Tuple[Array.Array]:
        self._check_inverse_input_shape(y)
        masked_y = jnp.where(sef._event_mask, y, 0.0)
        params = self._conditioner(masked_y, context)
        x0, log_d = self._inner_bijector(params).inverse_and_log_det(y)
        x = jnp.where(self._event_mask, x, y0)
        logdet = math.sum_last(
            jnp.where(self._event_mask, 0., log_d),
            self._event_ndims - self._inner_event_ndims
        )
        return x, logdet


class RQSplineFlow(nnx.Module):
    layers: nnx.List

    def __init__(self,
                 features_shape: tuple,
                 context_shape: tuple = 0,
                 n_dim: int,
                 n_context: int = 0,
                 n_transforms: int = 4,
                 hidden_dims: List[int] = dataclasses.field(default_factory=lambda: [64, 64]),
                 activation: str = "gelu",
                 n_bins: int = 8,
                 range_min: float = -1.0,
                 range_max: float = 1.0,
                 bijector_type: Callable = distrax.RationalQuadraticSpline(),
                 ):
        
        self.features_shape = features_shape
        self.n_bins = n_bins
        self.range_min = range_min
        self.range_max = range_max
        self.num_bijector_params = 3*self.n_bins + 1
        self.bijector_type = bijector_type
        self.n_transforms = n_transforms # Number of conditioner-bijector layers in the overall flow

        # Bijector transformation used in each layer of the flow
        def bijector_fn(params: Array):
            return self.bijector(params, self.range_min, self.range_max)

        mask = jnp.arange(0, np.prod(event_shape)) % 2
        mask = jnp.reshape(mask, event_shape)
        mask = mask.astype(bool)

        # First instantiate all the conditioners needed for #n_transform RQSpline transforms
        self.conditioners = [] 
        for t in range(self.n_transforms):
            self.conditioners.append(
                Conditioner(self.features_shape, self.context_shape, self.hidden_dims, self.num_bijector_params)
            )

        # Now instantiate all the compuling trandforms
        self.transforms = []
        for t in range(self.n_transforms):
            transforms.append(
                ConditionalMaskedCoupling(mask=mask, bijector=bijector_fn, conditioner=self.conditioner[t])
                )
            mask = jnp.logical_not(mask) # Alternating binary mask on the input features to the conditioner for coupling transform

        # Now stack all transforms as single distrax.bjector child class 
        # Note that the convention used here is such that the RQSpline transforms map from data features to the conditional context,
        # We need to invert it for probability evaluation
        self.bijector = 
        self.base_dist = distrax.MultivariateNormalDiag(jnp.zeros(features_shape), jnp.ones(features_shape))

    
    def __call__(self):
        for layer in self.layers:
            x = layer(x)
        return x




NameError: name 'Sequence' is not defined